Plant Oil

In [16]:
import os
import pandas as pd

root_folder = r"D:\Synthetic Data Gen\New experiments\Plant Oil" 
output_file = "combined_oil_dataset.csv"

all_data = []
measurement_number = 1

if os.path.exists(root_folder):
    for label in os.listdir(root_folder):
        brand_path = os.path.join(root_folder, label)
        
        if os.path.isdir(brand_path):
            for file_name in os.listdir(brand_path):
                file_path = os.path.join(brand_path, file_name)
                
                if not file_name.startswith(('.', '~$')) and file_name.endswith(('.csv', '.txt')):
                    df = pd.read_csv(file_path)
                    df['Measurement_Number'] = measurement_number
                    df['Label'] = label
                    
                    df = df[['Measurement_Number'] + [c for c in df.columns if c != 'Measurement_Number']]
                    all_data.append(df)
                    measurement_number += 1

    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)
        final_df.to_csv(output_file, index=False)
        
        # --- Statistics and Print Results (Statistics and Print Results) ---
        num_labels = final_df['Label'].nunique()
        total_meas = measurement_number - 1
        total_rows = len(final_df)
        
        avg_meas = total_meas // num_labels if num_labels else 0
        avg_rows = total_rows // total_meas if total_meas else 0
        
        print(f"Label : {num_labels}")
        print(f"Measurements (sessions) count {avg_meas}x{num_labels}(label) = {total_meas}")
        print(f"Datapoint {avg_rows} x {total_meas}(measurements) = {total_rows}")
        print()
    else:
        print()
else:
    print()

Label : 19
Measurements (sessions) count 70x19(label) = 1330
Datapoint 150 x 1330(measurements) = 199500

Saved at: combined_oil_dataset.csv


Brewed Vinegar

In [11]:
import os
import pandas as pd
import numpy as np

# 1. Khai báo các đường dẫn (Declare paths)
# Sử dụng tiền tố 'r' trước chuỗi để Windows hiểu đúng đường dẫn (Use 'r' prefix for raw string so Windows understands the path correctly)
root_folder = r"D:\Synthetic Data Gen\New experiments\Brewed Vinegar" 
output_file = "combined_vinegar_dataset.csv"

# Danh sách các cột cảm biến theo thứ tự (List of sensor columns in order)
sensor_columns = ['MQ136', 'MQ9B', 'MQ7B', 'MQ2', 'MQ8', 'MQ138', 'MQ137', 'MQ5']

# Tần số lấy mẫu (Sample rate) = 20 samples/second -> 1 sample = 0.05s
sample_interval = 1.0 / 20.0 

# Danh sách chứa tất cả các dataframe (List containing all dataframes)
all_dataframes = []
measurement_number = 1

label_to_indicator = {}
current_indicator = 0

# Kiểm tra xem đường dẫn thư mục gốc có tồn tại không (Check if root directory path exists)
if not os.path.exists(root_folder):
    print()
    print()
else:
    # 2. Quét qua các thư mục và file (Scan through directories and files)
    for brand_folder in os.listdir(root_folder):
        brand_path = os.path.join(root_folder, brand_folder)
        
        # Chỉ xử lý nếu nó là một thư mục (Only process if it is a directory (e.g., JSHS, LNGQ,...))
        if os.path.isdir(brand_path):
            label = brand_folder
            
            # Gán indicator cho label nếu chưa có (Assign an integer indicator to label if it doesn't exist)
            if label not in label_to_indicator:
                label_to_indicator[label] = current_indicator
                current_indicator += 1
                
            label_indicator = label_to_indicator[label]
            
            # Đọc các file dữ liệu bên trong thư mục của brand (Read data files inside the brand directory)
            for file_name in os.listdir(brand_path):
                file_path = os.path.join(brand_path, file_name)
                
                # Bỏ qua các file ẩn của hệ thống (Skip hidden system files like .DS_Store)
                if file_name.startswith('.'):
                    continue
                
                # Hỗ trợ đọc file .csv, .txt và .xlsx (Support reading .csv, .txt, and .xlsx files)
                if file_name.endswith(('.csv', '.txt', '.xlsx')):
                    try:
                        # Đọc file dữ liệu, thiết lập header=None vì file mẫu không có dòng header (Read data file, set header=None since the sample file has no header line)
                        if file_name.endswith('.xlsx'):
                            df = pd.read_excel(file_path, header=None)
                        else:
                            df = pd.read_csv(file_path, header=None)
                        
                        # Kiểm tra số lượng cột (Check the number of columns)
                        if df.shape[1] > len(sensor_columns):
                            # Lấy đúng 8 cột đầu tiên (Take exactly the first 8 columns)
                            df = df.iloc[:, :len(sensor_columns)]
                        elif df.shape[1] < len(sensor_columns):
                            print()
                            continue
                            
                        # Gán tên cột cho các cảm biến (Assign column names for the sensors)
                        df.columns = sensor_columns
                        
                        # Thêm cột Measurement number (Add Measurement number column)
                        df['Measurement_Number'] = measurement_number
                        
                        # Thêm cột Time (Add Time column (starts from 0, increments by 0.05s per row))
                        df['Time'] = np.arange(len(df)) * sample_interval
                        
                        # Thêm cột Label và Label Indicator (Add Label and Label Indicator columns)
                        df['Label'] = label
                        df['Label_Indicator'] = label_indicator
                        
                        # Đưa dataframe vào danh sách (Append dataframe to list)
                        all_dataframes.append(df)
                        
                        # Tăng measurement_number sau mỗi file (Increment measurement_number after each file to ensure each file is a separate measurement)
                        measurement_number += 1
                        
                    except Exception as e:
                        print()

    # 3. Gộp tất cả data và xuất ra file (Merge all data and export to file)
    if all_dataframes:
        # Nối tất cả các DataFrame lại với nhau (Concatenate all DataFrames together)
        final_dataset = pd.concat(all_dataframes, ignore_index=True)
        
        # Sắp xếp lại thứ tự cột cho logic dễ nhìn hơn (Reorder columns for logical visibility)
        cols_order = ['Measurement_Number', 'Time'] + sensor_columns + ['Label', 'Label_Indicator']
        final_dataset = final_dataset[cols_order]
        
        # Lưu ra file CSV tổng (Save to combined CSV file)
        final_dataset.to_csv(output_file, index=False)
        print()
        print()
        print(f"Processed: {measurement_number - 1}")
        print(f"Done}")
        print()
    else:
        print


--- HOÀN TẤT ---
Dữ liệu đã được gộp và lưu tại cùng thư mục chạy code với tên: combined_vinegar_dataset.csv
Tổng số file/phép đo (measurements) đã xử lý: 150
Tổng số dòng dữ liệu: 675000
Bảng ánh xạ Label Indicator: {'JSHS': 0, 'LNGQ': 1, 'SCBN': 2, 'SXDH': 3, 'SXLF': 4, 'TJTL': 5}


Coffee

In [19]:
import os
import pandas as pd
import numpy as np

root_folder = r"D:\Synthetic Data Gen\New experiments\Coffee" 
output_file = "combined_coffee_dataset.csv"

sensors = ['SP-12A', 'SP-31', 'TGS-813', 'TGS-842', 'SP-AQ3', 'TGS-823', 'ST-31', 'TGS-800']
all_data = []
measurement_number = 1

if os.path.exists(root_folder):
    for label in os.listdir(root_folder):
        brand_path = os.path.join(root_folder, label)
        if os.path.isdir(brand_path):
            for file_name in os.listdir(brand_path):
                if not file_name.startswith(('.', '~$')) and file_name.endswith(('.csv', '.txt')):
                    file_path = os.path.join(brand_path, file_name)
                    
                    df = pd.read_csv(file_path, sep='\t', header=None).dropna(axis=1, how='all')
                    
                    if df.shape[1] >= 8:
                        df = df.iloc[:, :8]
                        df.columns = sensors
                        
                        df.insert(0, 'Measurement_Number', measurement_number)
                        df.insert(1, 'Time', np.arange(len(df)))
                        df['Label'] = label
                        
                        all_data.append(df)
                        measurement_number += 1

    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)
        final_df.to_csv(output_file, index=False)
        
        num_labels = final_df['Label'].nunique()
        total_meas = measurement_number - 1
        avg_meas = total_meas // num_labels if num_labels else 0
        total_rows = len(final_df)
        avg_rows = total_rows // total_meas if total_meas else 0
        
        print(f"Label : {num_labels}")
        print(f"Measurements number {avg_meas}x{num_labels}(label) = {total_meas}")
        print(f"Datapoint {avg_rows}x{total_meas} = {total_rows}")
        print()
else:
    print()

Label : 3
Measurements number 19x3(label) = 58
Datapoint 300x58 = 17400

Saved at: combined_coffee_dataset.csv


In [20]:
import os
import pandas as pd
import numpy as np

root_folder = r"D:\Synthetic Data Gen\New experiments\Wine Spoilage" 
output_file = "combined_wine_dataset.csv"

sensors = ['Relative Humidity', 'Temperature', 'MQ-3_1', 'MQ-4_1', 'MQ-6_1', 'MQ-3_2', 'MQ-4_2', 'MQ-6_2']
all_data = []
measurement_number = 1

if os.path.exists(root_folder):
    for label in os.listdir(root_folder):
        brand_path = os.path.join(root_folder, label)
        if os.path.isdir(brand_path):
            for file_name in os.listdir(brand_path):
                if not file_name.startswith(('.', '~$')) and file_name.endswith(('.csv', '.txt')):
                    file_path = os.path.join(brand_path, file_name)
                    
                    df = pd.read_csv(file_path, sep='\t', header=None)
                    
                    if df.shape[1] == 8:
                        df.columns = sensors
                        df.insert(0, 'Measurement_Number', measurement_number)
                        df.insert(1, 'Time', np.arange(len(df)))
                        df['Label'] = label
                        
                        all_data.append(df)
                        measurement_number += 1

    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)
        final_df.to_csv(output_file, index=False)
        
        num_labels = final_df['Label'].nunique()
        total_meas = measurement_number - 1
        avg_meas = total_meas // num_labels if num_labels else 0
        total_rows = len(final_df)
        avg_rows = total_rows // total_meas if total_meas else 0
        
        print(f"Label : {num_labels}")
        print(f"Measurements number {avg_meas}x{num_labels}(label) = {total_meas}")
        print(f"Datapoint {avg_rows}x{total_meas} = {total_rows}")
        print()
else:
    print()

Label : 4
Measurements number 75x4(label) = 300
Datapoint 3330x300 = 999000

Saved at: combined_wine_dataset.csv


In [21]:
import os
import pandas as pd
import numpy as np

root_folder = r"D:\Synthetic Data Gen\New experiments\Chinese Wine" 
output_file = "combined_chinese_wine_dataset.csv"

all_data = []
measurement_number = 1

if os.path.exists(root_folder):
    for label in os.listdir(root_folder):
        brand_path = os.path.join(root_folder, label)
        if os.path.isdir(brand_path):
            for file_name in os.listdir(brand_path):
                if not file_name.startswith(('.', '~$')) and file_name.endswith(('.csv', '.txt')):
                    file_path = os.path.join(brand_path, file_name)
                    
                    df = pd.read_csv(file_path)
                    
                    # Thêm các cột metadata (Add metadata columns)
                    df.insert(0, 'Measurement_Number', measurement_number)
                    df.insert(1, 'Time', np.arange(len(df)))
                    df['Label'] = label
                    
                    all_data.append(df)
                    measurement_number += 1

    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)
        final_df.to_csv(output_file, index=False)
        
        num_labels = final_df['Label'].nunique()
        total_meas = measurement_number - 1
        avg_meas = total_meas // num_labels if num_labels else 0
        total_rows = len(final_df)
        avg_rows = total_rows // total_meas if total_meas else 0
        
        print(f"Label : {num_labels}")
        print(f"Measurements number {avg_meas}x{num_labels}(label) = {total_meas}")
        print(f"Datapoint {avg_rows}x{total_meas} = {total_rows}")
        print()
else:
    print()

Label : 10
Measurements number 90x10(label) = 900
Datapoint 180x900 = 162000

Saved at: combined_chinese_wine_dataset.csv


| Dataset Name | Labels | Measurements | Datapoints |
| :--- | :--- | :--- | :--- |
| `combined_vinegar_dataset.csv` | 6 | **150** *(25 x 6)* | **675,000** *(4500 x 150)* |
| `combined_oil_dataset.csv` | 19 | **1330** *(70 x 19)* | **199,500** *(150 x 1330)* |
| `combined_coffee_dataset.csv` | 3 | **58** *(19 x 3)* | **17,400** *(300 x 58)* |
| `combined_wine_dataset.csv` | 4 | **300** *(75 x 4)* | **999,000** *(3330 x 300)* |
| `combined_chinese_wine_dataset.csv` | 10 | **900** *(90 x 10)* | **162,000** *(180 x 900)* |